# Tạo dataset nhận diện vùng bảng Excel

Chỉ cần chọn **Run All**. Notebook không sửa `Data/Raw`, không tạo DocType và không đưa value thật vào `dataset.jsonl`.

In [1]:
from pathlib import Path
import importlib
import json
import subprocess
import sys

def find_module_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'dataset_builder.py').is_file() and (candidate / 'Data' / 'Raw').is_dir():
            return candidate
    raise FileNotFoundError('Không tìm thấy dataset_builder.py và Data/Raw.')

MODULE_ROOT = find_module_root()
assert sys.version_info[:3] == (3, 14, 7), f'Hãy chọn kernel Python 3.14.7; hiện tại là {sys.version.split()[0]}'
print('Module root:', MODULE_ROOT)
print('Python:', sys.version.split()[0])

Module root: C:\Users\hoang\Downloads\eform_btp\Module\AI Import
Python: 3.14.7


In [2]:
from importlib.metadata import PackageNotFoundError, version

try:
    dependency_ok = version('openpyxl') == '3.1.5'
except PackageNotFoundError:
    dependency_ok = False

if not dependency_ok:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(MODULE_ROOT / 'requirements.txt')])
    importlib.invalidate_caches()
import openpyxl
assert openpyxl.__version__ == '3.1.5'
print('openpyxl:', openpyxl.__version__)

openpyxl: 3.1.5


## Nhãn chuẩn đã đối chiếu

File 21a dưới đây đã được đối chiếu với `Sổ làm việc2.xlsx`: vùng cần render là source `A2:M8`, header `A2:M7`.

In [3]:
CURATED_LABELS = [
    {
        'source_file': 'Data/Raw/Sở Giáo dục và đào tạo_21a_TP_HTPLDN_Kết quả triển khai công tác hỗ trợ phá(...).xlsx',
        'sheet_index': 0,
        'tables': [
            {'top': 2, 'left': 1, 'bottom': 8, 'right': 13, 'header_bottom': 7}
        ],
    }
]

sys.path.insert(0, str(MODULE_ROOT)) if str(MODULE_ROOT) not in sys.path else None
from dataset_builder import BuildOptions, build_dataset, encode_record

OPTIONS = BuildOptions(
    module_root=MODULE_ROOT,
    augment=True,
    minimum_family_samples=12,
    maximum_augmentations_per_sheet=5,
)
report = build_dataset(OPTIONS, curated_labels=CURATED_LABELS, progress=True)

Đã kiểm tra SHA-256: 1/1232
Đã kiểm tra SHA-256: 200/1232
Đã kiểm tra SHA-256: 400/1232
Đã kiểm tra SHA-256: 600/1232
Đã kiểm tra SHA-256: 800/1232
Đã kiểm tra SHA-256: 1000/1232
Đã kiểm tra SHA-256: 1200/1232
Đã kiểm tra SHA-256: 1232/1232
Đã tạo nhãn: 1/929 workbook duy nhất
Đã tạo nhãn: 100/929 workbook duy nhất
Đã tạo nhãn: 200/929 workbook duy nhất
Đã tạo nhãn: 300/929 workbook duy nhất
Đã tạo nhãn: 400/929 workbook duy nhất
Đã tạo nhãn: 500/929 workbook duy nhất
Đã tạo nhãn: 600/929 workbook duy nhất
Đã tạo nhãn: 700/929 workbook duy nhất
Đã tạo nhãn: 800/929 workbook duy nhất
Đã tạo nhãn: 900/929 workbook duy nhất
Đã tạo nhãn: 929/929 workbook duy nhất
Hoàn tất: 929 sheet, 956 bảng, 976 augmentation recipes.


In [4]:
dataset_path = MODULE_ROOT / 'Data' / 'dataset.jsonl'
records = [json.loads(line) for line in dataset_path.read_text(encoding='utf-8').splitlines() if line.strip()]
assert report['ready_for_training'], report['parse_errors']
assert report['curated_label_count'] == len(CURATED_LABELS)
assert report['validation']['contains_no_doctype_field']
assert report['validation']['contains_no_raw_cell_values']

known = next(item for item in records if item['source_file'] == CURATED_LABELS[0]['source_file'])
assert known['tables'] == CURATED_LABELS[0]['tables']
features, targets = encode_record(MODULE_ROOT, known)
assert targets == CURATED_LABELS[0]['tables']
expected_cell_keys = {'row', 'column', 'token', 'value_type', 'is_negative', 'is_negative_sequence_row', 'bold', 'centered', 'border_count'}
assert all(set(cell) == expected_cell_keys for cell in features['cells'])

augmented_record = next(item for item in records if item['augmentation_recipes'])
_, augmented_targets = encode_record(MODULE_ROOT, augmented_record, 0)
recipe = augmented_record['augmentation_recipes'][0]
assert augmented_targets[0]['top'] == augmented_record['tables'][0]['top'] + recipe['row_shift']
assert augmented_targets[0]['left'] == augmented_record['tables'][0]['left'] + recipe['column_shift']

print(json.dumps({
    'ready_for_training': report['ready_for_training'],
    'raw_workbooks': report['raw_workbook_count'],
    'unique_workbooks': report['unique_workbook_count'],
    'sheet_samples': report['sheet_sample_count'],
    'tables': report['table_count'],
    'splits': report['split_sheet_counts'],
    'augmentation_recipes': report['augmentation_recipe_count'],
    'effective_training_samples': report['effective_training_sample_count'],
    'known_21a_target': known['tables'],
}, ensure_ascii=False, indent=2))

{
  "ready_for_training": true,
  "raw_workbooks": 1232,
  "unique_workbooks": 929,
  "sheet_samples": 929,
  "tables": 956,
  "splits": {
    "test_id": 130,
    "test_ood": 34,
    "train": 694,
    "validation": 71
  },
  "augmentation_recipes": 976,
  "effective_training_samples": 1670,
  "known_21a_target": [
    {
      "top": 2,
      "left": 1,
      "bottom": 8,
      "right": 13,
      "header_bottom": 7
    }
  ]
}


## Xem trước trực quan từ Raw

Cell dưới chọn cố định một mẫu đại diện của validation, train, test-id và test-ood rồi hiển thị value thật, header và merge giống một bảng preview. Value chỉ được đọc để hiển thị, không được ghi vào `dataset.jsonl`.

In [5]:
from html import escape
from random import Random
from IPython.display import HTML, display
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.utils.cell import range_boundaries

def show_table_preview(record, table_index=0, max_rows=12, max_columns=16):
    workbook = load_workbook(MODULE_ROOT / record['source_file'], data_only=False, read_only=False)
    try:
        sheet = workbook.worksheets[record['sheet_index']]
        table = record['tables'][table_index]
        visible_bottom = min(table['bottom'], table['top'] + max_rows - 1)
        visible_right = min(table['right'], table['left'] + max_columns - 1)
        anchors, covered = {}, set()
        for merged_range in sheet.merged_cells.ranges:
            left, top, right, bottom = range_boundaries(str(merged_range))
            clip_top, clip_left = max(top, table['top']), max(left, table['left'])
            clip_bottom, clip_right = min(bottom, visible_bottom), min(right, visible_right)
            if clip_top > clip_bottom or clip_left > clip_right:
                continue
            anchors[(clip_top, clip_left)] = (clip_bottom - clip_top + 1, clip_right - clip_left + 1)
            covered.update(
                (row, column)
                for row in range(clip_top, clip_bottom + 1)
                for column in range(clip_left, clip_right + 1)
                if (row, column) != (clip_top, clip_left)
            )
        html_rows = []
        for row in range(table['top'], visible_bottom + 1):
            html_cells = []
            for column in range(table['left'], visible_right + 1):
                if (row, column) in covered:
                    continue
                rowspan, colspan = anchors.get((row, column), (1, 1))
                value = sheet.cell(row, column).value
                value_html = escape('' if value is None else str(value)).replace('\n', '<br>')
                cell_class = 'eform-header' if row <= table['header_bottom'] else 'eform-value'
                html_cells.append(
                    f'<td class="{cell_class}" rowspan="{rowspan}" colspan="{colspan}">{value_html}</td>'
                )
            html_rows.append('<tr>' + ''.join(html_cells) + '</tr>')
        source_range = (
            f"{get_column_letter(table['left'])}{table['top']}:"
            f"{get_column_letter(table['right'])}{table['bottom']}"
        )
        clipped = visible_bottom < table['bottom'] or visible_right < table['right']
        title = (
            f"<h4>{escape(record['split'])} · {escape(sheet.title)} · {source_range}</h4>"
            f"<div class='eform-file'>{escape(record['source_file'])}</div>"
        )
        note = "<div class='eform-note'>Đang rút gọn preview; dữ liệu gốc không bị cắt.</div>" if clipped else ''
        style = '''<style>
          .eform-preview{border-collapse:collapse;margin:8px 0 20px;font-family:Arial;font-size:12px}
          .eform-preview td{border:1px solid #64748b;padding:5px;min-width:70px;max-width:220px;white-space:pre-wrap}
          .eform-header{background:#dbeafe;font-weight:600;text-align:center}
          .eform-value{background:#f8fafc}
          .eform-file,.eform-note{font-size:12px;color:#475569;margin:4px 0}
        </style>'''
        display(HTML(style + title + note + "<table class='eform-preview'>" + ''.join(html_rows) + '</table>'))
    finally:
        workbook.close()

rng = Random(20260913)
preview_records = [known]
for split_name in ('train', 'test_id', 'test_ood'):
    candidates = [
        item for item in records
        if item['split'] == split_name
        and item['tables']
        and item['tables'][0]['bottom'] > item['tables'][0]['header_bottom']
    ]
    preview_records.append(rng.choice(candidates))

for preview_record in preview_records:
    show_table_preview(preview_record)

## Hoàn tất

Khi các cell trên không báo lỗi và in `ready_for_training: true`, dataset nằm ở `Data/dataset.jsonl`; báo cáo nằm ở `Data/dataset_report.json`.